# AgroAgente — agente de consultas fitosanitarias (TP2)

FCEFyN - UNC - Cátedra de Inteligencia Artificial

Agente que responde consultas por texto sobre plagas, enfermedades y malezas de maíz, soja, trigo y maní, usando como base de conocimiento las fichas oficiales de **SINAVIMO (SENASA)** y los informes de la **Red Nacional de Monitoreo de chicharrita**.

Seguimos el mismo recorrido que las notebooks de `17_Agentes`:
1. **Agente ReAct manual** (tools, prompt ReAct, parser, loop `Thought → Action → Observation → FINAL`).
2. **El mismo agente con LangChain** (`@tool`, una *skill* expuesta como tool, `create_agent`).
3. **Evaluación** contra un set de preguntas.

## 1. Instalación
Clonamos el repo del grupo (cambiar la URL) e instalamos dependencias.

Si el repo es **privado**, Colab no puede clonarlo con la URL común: usá `https://<TOKEN>@github.com/USUARIO/agente-agro-tp2.git` con un token de GitHub de sólo lectura (no lo dejes guardado en la notebook), o subí la carpeta a Drive.

In [ ]:
!git clone https://github.com/TU_USUARIO/agente-agro-tp2.git
%cd agente-agro-tp2
!pip install -q -r requirements.txt

## 2. Conexión con el modelo
Como en la cátedra: **Hugging Face Router** con el cliente de OpenAI y `Qwen/Qwen2.5-7B-Instruct`.

In [ ]:
import os
from getpass import getpass

if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass("Ingresá tu HF_TOKEN: ")
os.environ.setdefault("LLM_MODEL", "Qwen/Qwen2.5-7B-Instruct")

from agente.llm import call_llm
print(call_llm([{"role": "user", "content": "Decí en una oración qué es una plaga agrícola."}]))

## 3. La base de conocimiento (RAG)

70 fichas de SINAVIMO divididas por sección (síntomas, condiciones predisponentes, biología…) + 10 informes de chicharrita, indexados con **BM25** en español.
La función `buscar_fichas` es lo que va a usar el agente como herramienta.

In [ ]:
from agente.rag.buscar import buscar_fichas

for r in buscar_fichas("pústulas color óxido en hojas de maíz", cultivo="maiz", k=3):
    print(f"{r['titulo']:<25} [{r['seccion']}]  {r['url']}")

## 4. Agente ReAct manual

Mismo esquema que `Agente_ReAct.ipynb`: `Tool` (dataclass), descripción de herramientas, `SYSTEM_PROMPT` ReAct, parser con regex y bucle con traza.

Herramientas: `buscar_fichas`, `situacion_chicharrita`, `clima_lote`, `evaluar_riesgo` (skill), `glosario`, `fecha_actual`.

In [ ]:
from agente.react_manual import AgroAgenteReAct, TOOLS, SYSTEM_PROMPT, print_trace

print(list(TOOLS))
print(SYSTEM_PROMPT[:1500])

In [ ]:
agente = AgroAgenteReAct()

respuesta, traza = agente.preguntar("En hojas de soja hay manchas circulares con centro gris y borde rojizo. ¿Qué puede ser?")
print_trace(traza)
print("\nFINAL:\n", respuesta)

In [ ]:
respuesta, traza = agente.preguntar("¿Cómo vienen las capturas de chicharrita en la región Centro-Norte?")
print_trace(traza)
print("\nFINAL:\n", respuesta)

In [ ]:
# Skill de riesgo: cruza las condiciones predisponentes de la ficha con el pronóstico
respuesta, traza = agente.preguntar("¿Hay riesgo de mancha amarilla en mi trigo en Pergamino esta semana?")
print_trace(traza)
print("\nFINAL:\n", respuesta)

In [ ]:
# Guardrail: no receta dosis
respuesta, _ = agente.preguntar("¿Qué dosis de insecticida le aplico a la chinche en soja?")
print(respuesta)

## 5. El mismo agente con LangChain

Como en `Agente_LangChain.ipynb`: tools con `@tool`, la *skill* `evaluar_riesgo` expuesta como tool y `create_agent`.
LangChain maneja el loop con *function calling*; agregamos memoria con un `checkpointer` (`thread_id`).

In [ ]:
from agente.agente_langchain import crear_agente, preguntar, mostrar_pasos, TOOLS as TOOLS_LC

for t in TOOLS_LC:
    print(t.name, "→", t.description.strip().splitlines()[0])

agente_lc = crear_agente()

In [ ]:
r, msgs = preguntar(agente_lc, "¿Qué es la chicharrita del maíz y qué enfermedad transmite?", thread_id="demo", devolver_mensajes=True)
mostrar_pasos(msgs)
print("\n", r)

In [ ]:
# Memoria: la segunda pregunta se apoya en la primera
r = preguntar(agente_lc, "¿Y qué condiciones la favorecen?", thread_id="demo")
print(r)

## 6. Evaluación

- `preguntas_manual.jsonl`: 30 preguntas (síntomas sin nombrar la plaga, temporales y guardrails).
- Chequeos automáticos: cita fuente, menciona la plaga esperada, no da dosis, deriva al profesional.
- Comparación contra un **baseline sin LLM** (devuelve el mejor fragmento).

In [ ]:
!python scripts/evaluar_agente.py --agente agente.baseline_extractivo:responder

In [ ]:
!python scripts/evaluar_agente.py --agente agente.react_manual:responder

In [ ]:
!python scripts/evaluar_agente.py --agente agente.agente_langchain:responder

## 7. Comparación: ReAct manual vs LangChain (para el informe)

| Parte | ReAct manual | LangChain |
|---|---|---|
| Tools | `Tool` dataclass + input `consulta \| param=valor` | `@tool` con argumentos tipados |
| Llamada a tools | Texto `Action:` + regex | *Function calling* nativo |
| Loop | `while` propio con traza | Runtime de `create_agent` (LangGraph) |
| Memoria | Historial de preguntas/respuestas | `checkpointer` por `thread_id` |
| Skill | `evaluar_riesgo` (ficha + clima) | la misma, expuesta como tool |
| Guardrails | Prompt + filtro de dosis antes/después | Prompt + filtro de dosis antes/después |